# Notebook 04 — Applying ∇ to "real" data

So far every example was a clean analytical formula. The whole point of
finite-difference operators is that they work on *measured* fields where no
formula is available.

This notebook applies the three operators to two scenarios:

1. **Terrain elevation** (a synthetic Digital Elevation Model). The gradient
   gives slope direction — exactly what GIS tools call *aspect*.
2. **2D fluid flow** (a superposition of a point source and a vortex).
   Divergence isolates the source; curl isolates the vortex.

The data is synthetic — generated in this notebook — so you can swap in real
data later (a USGS DEM, a NetCDF wind grid, etc.) and the rest of the
analysis stays identical.


In [ ]:
# Make src/ importable from inside notebooks/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline

from src import operators as ops
from src import visualizers as viz


## 1. Terrain — gradient as slope direction

In [ ]:
# Build a synthetic landscape: a sum of Gaussian "hills" plus a gentle ramp
np.random.seed(42)

x_vals = np.linspace(0, 10, 300)
y_vals = np.linspace(0, 10, 300)
X, Y = np.meshgrid(x_vals, y_vals)
dx, dy = x_vals[1] - x_vals[0], y_vals[1] - y_vals[0]

def gaussian_hill(cx, cy, height, width):
    return height * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * width**2))

elevation = (
    gaussian_hill(3, 3, 800, 1.2)    # tall narrow peak
    + gaussian_hill(7, 4, 500, 1.8)  # broader hill
    + gaussian_hill(5, 7, 300, 2.5)  # gentle rise
    + 30 * X / 10                    # gentle westerly slope
)

# Save it so it can be loaded by other notebooks if you want
np.save("../data/elevation.npy", elevation)
print(f"Elevation grid shape: {elevation.shape}")
print(f"Range: {elevation.min():.1f} m to {elevation.max():.1f} m")


In [ ]:
# Compute the gradient — this is the slope vector at every grid point
dz_dx, dz_dy = ops.gradient_2d(elevation, dx, dy)

# Slope magnitude (rise per unit horizontal distance)
slope = np.hypot(dz_dx, dz_dy)

# Aspect: the compass direction the slope faces, in degrees from north (0=N)
aspect_rad = np.arctan2(-dz_dx, -dz_dy)  # negative because we want downhill
aspect_deg = (np.degrees(aspect_rad) + 360) % 360

print(f"Max slope: {slope.max():.2f} m/m  (≈ {np.degrees(np.arctan(slope.max())):.1f}°)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elevation with gradient arrows
ax = axes[0]
c = ax.contourf(X, Y, elevation, levels=25, cmap="terrain")
ax.contour(X, Y, elevation, levels=15, colors="black", linewidths=0.3, alpha=0.5)
step = 15
ax.quiver(X[::step, ::step], Y[::step, ::step],
          dz_dx[::step, ::step], dz_dy[::step, ::step],
          color="white", scale=2000, width=0.003)
ax.set_title("Elevation with ∇(elev) — uphill arrows")
ax.set_aspect("equal")
fig.colorbar(c, ax=ax, label="m")

# Slope magnitude
ax = axes[1]
c = ax.contourf(X, Y, slope, levels=25, cmap="magma")
ax.set_title("Slope magnitude  |∇(elev)|")
ax.set_aspect("equal")
fig.colorbar(c, ax=ax, label="m/m")

# Aspect (downhill direction)
ax = axes[2]
c = ax.contourf(X, Y, aspect_deg, levels=36, cmap="hsv")
ax.set_title("Aspect (downhill compass bearing, 0°=N)")
ax.set_aspect("equal")
fig.colorbar(c, ax=ax, label="deg")

plt.tight_layout()
plt.show()


**Reading this.** The gradient does what GIS terrain analysis does
under the hood:

- *Left*: arrows point uphill, longest where the slope is steepest.
- *Middle*: the magnitude alone is the *slope map* — useful for
  trafficability, erosion modeling, ski runs.
- *Right*: rotating the gradient by 180° gives *aspect* — which compass
  direction each hillside faces. This is what determines how much sun a slope
  gets, where snow lingers, etc.

You can replace `elevation` with a real `.tif` DEM loaded via `rasterio` and
get the same maps.

## 2. Fluid flow — divergence isolates sources, curl isolates vortices

In [ ]:
# Build a flow field by superposing two textbook patterns:
# - a point source at (-1, 0)
# - a point vortex at (+1, 0)

x_vals = np.linspace(-3, 3, 300)
y_vals = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x_vals, y_vals)
dx, dy = x_vals[1] - x_vals[0], y_vals[1] - y_vals[0]

def point_source(cx, cy, strength=1.0, eps=0.05):
    rx, ry = X - cx, Y - cy
    r2 = rx**2 + ry**2 + eps  # eps avoids division by zero at the singularity
    return strength * rx / r2, strength * ry / r2

def point_vortex(cx, cy, strength=1.0, eps=0.05):
    rx, ry = X - cx, Y - cy
    r2 = rx**2 + ry**2 + eps
    return -strength * ry / r2, strength * rx / r2

src_x, src_y = point_source(-1.0, 0.0, strength=1.5)
vor_x, vor_y = point_vortex(1.0, 0.0, strength=1.5)

Fx = src_x + vor_x
Fy = src_y + vor_y


In [ ]:
div = ops.divergence_2d(Fx, Fy, dx, dy)
curl = ops.curl_2d(Fx, Fy, dx, dy)

fig = viz.plot_vector_with_divergence(
    X, Y, Fx, Fy, div,
    title="Divergence highlights the source on the left",
)
plt.show()

fig = viz.plot_vector_with_curl(
    X, Y, Fx, Fy, curl,
    title="Curl highlights the vortex on the right",
)
plt.show()


**Reading this.** The combined flow looks like a tangled mess of
streamlines, but the operators cleanly separate the two contributions:

- The **divergence** plot lights up at $(-1, 0)$, where the source lives.
  The vortex contributes nothing — it has zero divergence (it's solenoidal).
- The **curl** plot lights up at $(+1, 0)$, where the vortex lives. The
  source contributes nothing — it has zero curl (it's irrotational).

This is exactly how meteorologists pull useful structure out of measured wind
fields: divergence flags convergence/divergence zones (storm formation,
weather fronts); curl (called *vorticity* in this domain) flags rotating
systems like cyclones and tornadoes.


## 3. The Helmholtz decomposition (sneak peek)

The fact that the source had no curl and the vortex had no divergence is not
an accident. **Helmholtz's theorem** says any sufficiently nice vector field
can be split into:

$$\mathbf{F} = \nabla \phi + \nabla \times \mathbf{A}$$

…a **gradient part** (carries all the divergence, no curl) plus a **curl
part** (carries all the curl, no divergence). The example above is already
in that form. This decomposition is the mathematical backbone of fluid
dynamics, electromagnetism, and computer graphics fluid simulations.

If you want to keep going: try implementing the inverse operation —
*reconstructing* a field from its divergence and curl by solving Poisson
equations.
